# Code Runner Local Execute (Non-Isolated) Example

이 노트북은 `code_runner` SubAgent에만 **로컬 실행형 backend**를 붙이는 예제입니다.

중요: 이 방식은 실행 격리(sandbox isolation)가 없습니다.
- 호스트 머신에서 직접 프로세스를 실행합니다.
- trusted local development 환경에서만 권장됩니다.


## 기존 데모 노트북과 차이

비교 대상: `docs/code_runner_sandbox_subagent_example.ipynb`

| 항목 | 기존 데모 노트북 | 이 노트북 |
|---|---|---|
| 실행 방식 | `subprocess(..., shell=True)` | `subprocess(..., shell=False)` |
| 명령 제어 | 사실상 없음 | 허용 명령(prefix allowlist) 적용 |
| 위험 명령 차단 | 없음 | 일부 토큰/패턴 차단 |
| 작업 디렉터리 | 고정 없음 | `/research_workspace/scratch`로 고정 |
| 실행 환경변수 | 기본 환경 상속 | 비밀키 없는 `.env.code_runner.local`만 로드 |
| 목적 | 구조 시연 | 로컬에서 좀 더 안전한 운영형 템플릿 |
| 격리 수준 | 비격리 | 비격리 (동일) |

정리: 둘 다 보안 sandbox는 아니고, 이 노트북은 로컬 실행시 가드레일을 추가한 버전입니다.

In [1]:
from __future__ import annotations

import os
from pathlib import Path
import shlex
import subprocess

from deepagents import create_deep_agent
from deepagents.backends import CompositeBackend, FilesystemBackend, StateBackend
from deepagents.backends.protocol import ExecuteResponse, SandboxBackendProtocol
from langchain.tools import ToolRuntime
from langchain_openai import ChatOpenAI


import sys
from pathlib import Path

ROOT = Path("/Users/jungj2/Dropbox/Metadata/aiagent/DeepMAST")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.agents.drug_target.tools import (
    sequential_thinking,
)

In [2]:
class GuardedLocalExecBackend(FilesystemBackend, SandboxBackendProtocol):
    """Local non-isolated execute backend with guardrails."""

    ALLOWED_PREFIXES = {"python", "python3", "uv", "pip", "pip3"}
    BLOCKED_TOKENS = {"&&", "||", ";", "|", "`", "$()"}

    def __init__(
        self,
        root_dir: str | Path,
        *,
        virtual_mode: bool = True,
        timeout_sec: int = 90,
        work_dir: str = "/research_workspace/scratch",
        env_file: str | Path = ".env.code_runner.local",
    ) -> None:
        super().__init__(root_dir=root_dir, virtual_mode=virtual_mode, max_file_size_mb=20)
        self.timeout_sec = timeout_sec
        self.work_dir = work_dir
        self.exec_cwd = self._resolve_path(work_dir)
        self.exec_cwd.mkdir(parents=True, exist_ok=True)
        self.exec_env = self._build_exec_env(env_file)
        self._backend_id = f"guarded-local-exec:{self.cwd}"

    @property
    def id(self) -> str:
        return self._backend_id

    def _build_exec_env(self, env_file: str | Path) -> dict[str, str]:
        safe_env = {
            "PATH": os.environ.get("PATH", ""),
            "HOME": os.environ.get("HOME", ""),
            "PYTHONPATH": os.environ.get("PYTHONPATH", ""),
        }

        env_path = Path(env_file)
        if not env_path.is_absolute():
            env_path = self.cwd / env_path

        if not env_path.exists():
            return safe_env

        forbidden_markers = ("KEY", "TOKEN", "SECRET", "PASSWORD")
        for raw in env_path.read_text(encoding="utf-8").splitlines():
            line = raw.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            key = key.strip()
            value = value.strip().strip('\"').strip("'")

            upper_key = key.upper()
            if any(marker in upper_key for marker in forbidden_markers):
                raise ValueError(
                    f"Forbidden secret-like key in {env_path}: {key}. Use non-secret env only."
                )

            safe_env[key] = value

        return safe_env

    def _validate_command(self, command: str) -> tuple[bool, str]:
        text = command.strip()
        if not text:
            return False, "Empty command"

        for token in self.BLOCKED_TOKENS:
            if token in text:
                return False, f"Blocked token detected: {token}"

        try:
            parts = shlex.split(text)
        except ValueError as e:
            return False, f"Invalid command parsing: {e}"

        if not parts:
            return False, "Command parse result is empty"

        if parts[0] not in self.ALLOWED_PREFIXES:
            allowed = ", ".join(sorted(self.ALLOWED_PREFIXES))
            return False, f"Command prefix '{parts[0]}' is not allowed. Allowed: {allowed}"

        return True, ""

    def execute(self, command: str) -> ExecuteResponse:
        ok, reason = self._validate_command(command)
        if not ok:
            return ExecuteResponse(output=f"Blocked command: {reason}", exit_code=126, truncated=False)

        args = shlex.split(command)

        try:
            completed = subprocess.run(
                args,
                shell=False,
                cwd=str(self.exec_cwd),
                env=self.exec_env,
                capture_output=True,
                text=True,
                timeout=self.timeout_sec,
            )
            output = (completed.stdout or "") + (completed.stderr or "")
            truncated = False
            if len(output) > 12_000:
                output = output[:12_000] + "\n...[truncated]"
                truncated = True

            return ExecuteResponse(output=output, exit_code=completed.returncode, truncated=truncated)
        except subprocess.TimeoutExpired:
            return ExecuteResponse(
                output=f"Command timed out after {self.timeout_sec}s",
                exit_code=124,
                truncated=False,
            )


In [3]:
# 모델 (환경에 맞게 교체 가능)
model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

BASE_DIR = ROOT
RESEARCH_WORKSPACE_DIR = BASE_DIR / "tests/drugtarget"
CODE_RUNNER_ENV_FILE = BASE_DIR / ".env.code_runner.local"

# 메인 에이전트: execute 미사용
main_fs_backend = FilesystemBackend(root_dir=RESEARCH_WORKSPACE_DIR, virtual_mode=True, max_file_size_mb=20)

def main_backend_factory(rt: ToolRuntime):
    return CompositeBackend(
        default=StateBackend(rt),
        routes={"/": main_fs_backend},
    )

# code_runner 전용 local-exec backend
code_runner_backend = GuardedLocalExecBackend(
    root_dir=BASE_DIR,
    virtual_mode=True,
    timeout_sec=90,
    work_dir="/tests/drugtarget",
    env_file=CODE_RUNNER_ENV_FILE,
)

def code_runner_backend_factory(rt: ToolRuntime):
    _ = rt
    return code_runner_backend

CODE_RUNNER_PROMPT = """
You are code_runner.
- Create small Python files under /research_workspace/scratch
- Run only Python-related commands
- Always report command, exit code, output summary
"""

code_runner_runnable = create_deep_agent(
    model=model,
    tools=[],
    system_prompt=CODE_RUNNER_PROMPT,
    backend=code_runner_backend_factory,
    skills=["/skills/"],
)

code_runner_subagent = {
    "name": "code_runner",
    "description": "Write/modify Python files and run them in guarded local execute backend.",
    "runnable": code_runner_runnable,
}

MAIN_PROMPT = """
You are the orchestrator.
Use code_runner only for execution tasks.
"""

agent = create_deep_agent(
    model=model,
    tools=[sequential_thinking],
    system_prompt=MAIN_PROMPT,
    backend=main_backend_factory,
    subagents=[code_runner_subagent],
)


In [5]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Use subagent_type='code_runner'. Create tests/drugtarget/demo_calc.py, "
                    "run it with python3, and summarize command + exit code + output."
                ),
            }
        ]
    }
)

print(result["messages"][-1].content)


It appears that although the file /tests/drugtarget/demo_calc.py exists and I can read its contents, attempts to run it with python3 result in an error stating the file does not exist. This might be due to environment or permission issues in the execution context.

Would you like me to try running the script content directly in a code runner environment instead of running the file? This would allow me to execute the code and capture the output without relying on the file system execution.
